## RAG Application Using TypeSense

In [1]:
import typesense

In [4]:
clinet = typesense.Client({
    'nodes': [{
        'host':'3ntfghb7mqex1cdyp-1.a1.typesense.net',                                             
        ## For Typesense Cloud use xxx.a1.typesense.net. For own machine hosting use 'localhost',
        'port':'443',                                             ## For localhost it will be '8108',
        'protocol': 'https'                                  ## For Typesense Cloud use 'https'. For localhost it will be 'http',
    }],
    'api_key': 'aN58AZBDWD1d5t9vRZ1MwWAqYG7ED162',
    'connection_timeout_seconds': 2
})

book_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string' },
        {'name': 'authors', 'type': 'string[]', 'facet': True },
        {'name': 'publication_year', 'type': 'int32', 'facet': True },
        {'name': 'ratings_count', 'type': 'int32' },
        {'name': 'average_rating', 'type': 'float' },
    ],
    'default_sorting_field': 'ratings_count'
}
print(clinet.collections.create(book_schema))

{'created_at': 1762927762, 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'average_rating', 'optional': False, 'sort': True, 'stem': Fal

In [5]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    books = jsonl_file.read()
    clinet.collections['books'].documents.import_(books)

In [7]:
search_parameters = {
    'q': 'The Great Gatsby',
    'query_by': 'title,authors',
    'sort_by': 'ratings_count:desc'
}
search_results = clinet.collections['books'].documents.search(search_parameters)
search_results

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['F. Scott Fitzgerald'],
    'average_rating': 3.89,
    'id': '5',
    'image_url': 'https://images.gr-assets.com/books/1490528560m/4671.jpg',
    'publication_year': 1925,
    'ratings_count': 2683664,
    'title': 'The Great Gatsby'},
   'highlight': {'title': {'matched_tokens': ['The', 'Great', 'Gatsby'],
     'snippet': '<mark>The</mark> <mark>Great</mark> <mark>Gatsby</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['The', 'Great', 'Gatsby'],
     'snippet': '<mark>The</mark> <mark>Great</mark> <mark>Gatsby</mark>'}],
   'text_match': 1736172819517538425,
   'text_match_info': {'best_field_score': '3315704398080',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1736172819517538425',
    'tokens_matched': 3,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name': 'books',
  'first_q': 'The Great Gats

In [8]:
search_parameters = {
    'q': 'Harry Potter',
    'query_by': 'title',
    'filter_by': 'publication_year:>1999 && average_rating:>4.0',
    'sort_by': 'ratings_count:desc'
}
clinet.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 10,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.53,
    'id': '24',
    'image_url': 'https://images.gr-assets.com/books/1361482611m/6.jpg',
    'publication_year': 2000,
    'ratings_count': 1753043,
    'title': 'Harry Potter and the Goblet of Fire'},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Goblet of Fire'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Goblet of Fire'}],
   'text_match': 1157451471441100921,
   'text_match_info': {'best_field_score': '2211897868288',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441100921',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rat

In [9]:
search_parameters = {
    'q': 'experimental',
    'query_by': 'title',
    'facet_by': 'publication_year, authors',
    'sort_by': 'ratings_count:desc'
}
clinet.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': '2012',
     'value': '2012'},
    {'count': 1, 'highlighted': '2005', 'value': '2005'},
    {'count': 1, 'highlighted': '1940', 'value': '1940'}],
   'field_name': 'publication_year',
   'sampled': False,
   'stats': {'avg': 1985.6666666666667,
    'max': 2012.0,
    'min': 1940.0,
    'sum': 5957.0,
    'total_values': 3}},
  {'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152

In [ ]:
### LangChain + Typesnse + Groq LLM + RAG Application
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense      ##using Typesense as Vector Store in local or cloud
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings # <--- CORRECT
from langchain_groq import ChatGroq

In [5]:
import os
os.environ['Groq_API_KEY'] = 'Groq_API_KEY'  


In [6]:
loader = TextLoader('test.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Created a chunk of size 1015, which is longer than the specified 1000
C:\Users\niazs\AppData\Local\Temp\ipykernel_31992\181114308.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [10]:
docsearc=Typesense.from_documents(
    texts,
    embeddings,
    typesense_client_params={
        "host": "3ntfghb7mqex1cdyp-1.a1.typesense.net",
        "port": "443",
        "protocol": "https",
        "typesense_api_key": "aN58AZBDWD1d5t9vRZ1MwWAqYG7ED162",
        "typesense_collection_name": "lang-chain"

    },
)


In [17]:
query="What do you know about the issue of algorithmic bias and fairness of AI?"
docs=docsearc.similarity_search(query)
docs[0].page_content

'However, the rapid progress of AI has outpaced the development of ethical frameworks and regulatory oversight, creating a host of critical \nsocietal dilemmas. One of the most pressing concerns is the issue of algorithmic bias and fairness. AI systems are trained on historical data, \nand if that data reflects existing societal inequitiesâ€”such as racial or gender biasâ€”the resulting algorithms will not only replicate but often \namplify that discrimination in areas like hiring, lending, and criminal justice risk assessment. This "black box" problem, where complex deep \nlearning models lack transparency, makes it difficult to ascertain how a decision was reached or who should be held accountable when an AI \nsystem causes harm or makes a discriminatory decision.'